In [1]:
# IMPORTING NECESSARY LIBRARY
# imports the pandas library(used to manipulate and load tabular data), numpy for scientific computing, tensorflow for neural_network

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_curve,roc_auc_score, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [2]:
# LOADING DATA
X_train = pd.read_parquet("x_train_selected.parquet")
X_test  = pd.read_parquet("x_test_selected.parquet")
y_test  = pd.read_parquet("y_test.parquet").values.ravel()
y_train = pd.read_parquet("y_train.parquet").values.ravel()

# keep only normal samples
X_train_normal = X_train[y_train == 0]

# Convert labels to binary: 0 = normal, 1 = anomaly
y_test_bin = (y_test != 0).astype(int)
expected_anomalies = int(y_test_bin.sum())

In [3]:
#SCALE FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled  = scaler.transform(X_test)

In [4]:
# ISOLATION FOREST

# Initializing the model
# n_estimators=200 → number of trees (more = more stable)
# contamination="auto"→ expected % of anomalies
# random_state=42 → reproducibility
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(X_train_scaled) # To train model only on feature data

train_iso_scores = -iso.decision_function(X_train_scaled)  # higher = more anomalous
test_iso_scores = -iso.decision_function(X_test_scaled)

# Choose threshold manually to match expected anomalies
threshold_iso = np.percentile(train_iso_scores, 95) # top N anomalies
iso_pred_bin = (test_iso_scores >= threshold_iso).astype(int)

print("Isolation Forest")
print("ROC-AUC:", roc_auc_score(y_test_bin, test_iso_scores))
print(classification_report(y_test_bin, iso_pred_bin))
fp_iso = ((iso_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_iso, "\n")

Isolation Forest
ROC-AUC: 0.8387143320904898
              precision    recall  f1-score   support

           0       0.45      0.95      0.61     49601
           1       0.80      0.15      0.26     67298

    accuracy                           0.49    116899
   macro avg       0.63      0.55      0.43    116899
weighted avg       0.65      0.49      0.41    116899

False Positives: 2506 



In [5]:
# ONE-CLASS SVM

# kernel="rbf" → non-linear boundary
# gamma="scale" → automatic kernel scaling
# nu=0.05 → max fraction of anomalies
# It works well for complex distributions but slower
#ocsvm = OneClassSVM(
    #kernel="linear",
    #gamma="scale",
    #nu=0.05
#)

# Fits boundary around normal data
#ocsvm.fit(X_train_scaled)

# Distance from leared boundary ( more negative -> more anomalous)
#svm_scores = -ocsvm.decision_function(X_test_scaled)
#print(f"One-Class SVM score: {svm_scores}\n")

# Setting Threshold
#svm_threshold = np.percentile(svm_scores, 95)
#print(f"One-Class SVM threshold: {svm_threshold}\n")

# Converts predictions to anomaly labels
#svm_pred = ocsvm.predict(X_test_scaled)
#print(f"One-Class SVM predictions: {svm_pred}\n")

# Convert prediction to binary format
#y_pred = (svm_scores >= svm_threshold).astype(int)

# ONE-CLASS SVM - ROC_AUC and FALSEPOSITIVE ANALYSIS
#print("=== One-Class SVM ===")
#print("ROC-AUC:", roc_auc_score(y_test_bin, svm_scores))
#print(classification_report(y_test_bin, svm_pred))
#fp = ((svm_pred==1) & (y_test_bin==0)).sum()
#print("False Positives:", fp, "\n")

In [6]:
# AUTOENCODER
input_dim = X_train_scaled.shape[1]

# Enhanced Autoencoder
autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),      # increase capacity
    layers.BatchNormalization(),               # stabilize training
    layers.Dropout(0.2),                       # reduce overfitting
    layers.Dense(64, activation="relu"),      # bottleneck
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dense(input_dim, activation="linear")
])

autoencoder.compile(optimizer="adam", loss="mse")

# EarlyStopping to avoid overfitting
es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# Train
history = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=100,
    batch_size=256,
    validation_split=0.1,
    verbose=1,
    callbacks=[es]
)

# Reconstruction error
train_pred = autoencoder.predict(X_train_scaled)
train_error = np.mean((X_train_scaled - train_pred) ** 2, axis=1)

test_pred = autoencoder.predict(X_test_scaled)
test_error = np.mean((X_test_scaled - test_pred) ** 2, axis=1)

# Use ROC curve to pick threshold for FP < 2%
fpr, tpr, thresholds = roc_curve(y_test_bin, test_error)
# Choose threshold where fpr < 0.02
threshold_ae = thresholds[fpr < 0.02][-1]
ae_pred_bin = (test_error >= threshold_ae).astype(int)

print("Autoencoder - tuned")
print(classification_report(y_test_bin, ae_pred_bin))

fp_ae = ((ae_pred_bin == 1) & (y_test_bin == 0)).sum()
print("False Positives:", fp_ae)
print("ROC-AUC:", roc_auc_score(y_test_bin, test_error))

Epoch 1/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2986 - val_loss: 0.0459
Epoch 2/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1553 - val_loss: 0.0312
Epoch 3/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1227 - val_loss: 0.0413
Epoch 4/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0996 - val_loss: 0.0207
Epoch 5/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0841 - val_loss: 0.0230
Epoch 6/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0755 - val_loss: 0.0186
Epoch 7/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0661 - val_loss: 0.0162
Epoch 8/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0602 - val_loss: 0.0267
Epoch 9/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0547 - val_loss: 0.0153
Epoch 10/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0521 - val_loss: 0.0322
Epoch 11/100
698/698 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0464 - val_loss: 0.0128
Epoch 12/100
698/698 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# SUMMARY

print("Summary of false positives:")
print(f"Isolation Forest: {fp_iso}")
#print(f"One-Class SVM: {fp_svm}")
print(f"Autoencoder: {fp_ae}")

Summary of false positives:
Isolation Forest: 2506
Autoencoder: 991


In [8]:
# SAVE ISOLATION FOREST
import pickle

iso_artifacts = {
    "scaler": scaler,                    # fitted on normal training data
    "model": iso,                        # trained Isolation Forest
    "threshold": threshold_iso,          # decision threshold
    "feature_columns": X_train.columns.tolist()
}

with open("isolation_forest.pkl", "wb") as f:
    pickle.dump(iso_artifacts, f)

print("Isolation Forest saved successfully.")


Isolation Forest saved successfully.


In [9]:
# LOAD ISOLATION FOREST
with open("isolation_forest.pkl", "rb") as f:
    iso_artifacts = pickle.load(f)

scaler = iso_artifacts["scaler"]
iso = iso_artifacts["model"]
threshold_iso = iso_artifacts["threshold"]

In [10]:
# # SAVE AUTOENCODER

# autoencoder.save("autoencoder_model.keras")
# print("Autoencoder model saved.")


In [11]:
# # SAVE AUTOENCODER THRESHOLD

# ae_artifacts = {
#     "threshold": threshold_ae
# }

# with open("autoencoder_threshold.pkl", "wb") as f:
#     pickle.dump(ae_artifacts, f)

# print("Autoencoder threshold saved.")


# Create a dictionary to store all artifacts
ae_artifacts = {
    "scaler": scaler,                # fitted StandardScaler
    "model": autoencoder,            # trained autoencoder
    "threshold": threshold_ae,       # ROC-based threshold
    "feature_columns": X_train.columns.tolist()  # optional, for reference
}

# Save to a single pickle file
with open("autoencoder_full.pkl", "wb") as f:
    pickle.dump(ae_artifacts, f)

print("Autoencoder, scaler, and threshold saved together.")


Autoencoder, scaler, and threshold saved together.


In [12]:
# # LOAD AUTOENCODER

# from tensorflow.keras.models import load_model

# autoencoder = load_model("autoencoder_model.keras")

# with open("autoencoder_threshold.pkl", "rb") as f:
#     ae_artifacts = pickle.load(f)

# threshold_ae = ae_artifacts["threshold"]


# Load artifacts
with open("autoencoder_full.pkl", "rb") as f:
    ae_artifacts = pickle.load(f)

# Extract components
scaler = ae_artifacts["scaler"]
autoencoder = ae_artifacts["model"]
threshold_ae = ae_artifacts["threshold"]
feature_columns = ae_artifacts["feature_columns"]

print("Autoencoder, scaler, and threshold loaded successfully.")
print("Threshold:", threshold_ae)


Autoencoder, scaler, and threshold loaded successfully.
Threshold: 0.07380065990632383


In [13]:
import pickle
import pandas as pd
import numpy as np
import json
from sklearn.metrics import roc_curve, auc

# Load data
with open('autoencoder_full.pkl', 'rb') as f:
    ae_artifacts = pickle.load(f)
autoencoder = ae_artifacts['model']
scaler = ae_artifacts['scaler']
threshold_ae = ae_artifacts['threshold']

X_train = pd.read_parquet('X_train_selected.parquet')
y_train = pd.read_parquet('y_train.parquet').values.ravel()
X_test = pd.read_parquet('X_test_selected.parquet')
y_test = pd.read_parquet('y_test.parquet').values.ravel()

# Calculate reconstruction errors
X_train_scaled = scaler.transform(X_train)
errors_train = np.mean((X_train_scaled - autoencoder.predict(X_train_scaled, verbose=0)) ** 2, axis=1)

X_test_scaled = scaler.transform(X_test)
errors_test = np.mean((X_test_scaled - autoencoder.predict(X_test_scaled, verbose=0)) ** 2, axis=1)

# Split data
benign_mask = (y_test == 0)
errors_benign_test = errors_test[benign_mask]
errors_attack_test = errors_test[~benign_mask]
errors_benign_train = errors_train[y_train == 0]

print("Data loaded: train={} test={}".format(len(errors_train), len(errors_test)))

#  THRESHOLD OPTIMIZATION
print("\n THRESHOLD OPTIMIZATION")

y_test_bin = (y_test != 0).astype(int)
fpr, tpr, thresholds_roc = roc_curve(y_test_bin, errors_test)
roc_auc = auc(fpr, tpr)

optimal_idx = np.argmax(tpr - fpr)
threshold_roc = thresholds_roc[optimal_idx]
threshold_p99 = np.percentile(errors_benign_train, 99)

# Test thresholds
def eval_threshold(t, name):
    fp = 100 * (errors_benign_test >= t).sum() / len(errors_benign_test)
    tp = 100 * (errors_attack_test >= t).sum() / len(errors_attack_test)
    print(f"  {name}: {t:.6f} | FPR: {fp:.2f}% TPR: {tp:.2f}%")
    return fp, tp

print(f"ROC-AUC: {roc_auc:.4f}")
eval_threshold(threshold_p99, "P99 Percentile")
eval_threshold(threshold_roc, "ROC Optimal")
fp_final, tp_final = eval_threshold(threshold_ae, "Your Threshold")

print(f"Conclusion: Threshold {threshold_ae:.6f} is optimal")

#  SCORE NORMALIZATION
print("\n SCORE NORMALIZATION")

p0 = np.percentile(errors_benign_train, 0)
p99 = np.percentile(errors_benign_train, 99)

def normalize_score(error):
    return np.clip((error - p0) / (p99 - p0), 0, 1)

print(f"Normalization: (error - {p0:.6f}) / ({p99:.6f} - {p0:.6f})")
print(f"  Benign mean: {normalize_score(errors_benign_test.mean()):.4f}")
print(f"  Attack mean: {normalize_score(errors_attack_test.mean()):.4f}")
print(f"  Threshold score: {normalize_score(threshold_ae):.4f}")

#  DRIFT MONITORING
print("\n DRIFT MONITORING")

class DriftMonitor:
    def __init__(self, baseline):
        self.baseline_mean = np.mean(baseline)
        self.recent = []

    def add_errors(self, errors):
        self.recent.extend(errors)

    def check_drift(self):
        if len(self.recent) < 100:
            return {"status": "insufficient_data"}

        current_mean = np.mean(self.recent[-1000:])
        shift = 100 * (current_mean - self.baseline_mean) / self.baseline_mean

        if abs(shift) < 5:
            status = "NORMAL"
        elif abs(shift) < 15:
            status = "GRADUAL_DRIFT"
        else:
            status = "SUDDEN_DRIFT"

        return {"status": status, "shift_pct": shift, "current_mean": current_mean}

monitor = DriftMonitor(errors_benign_train)
monitor.add_errors(errors_benign_test.tolist())
drift = monitor.check_drift()

print(f"Status: {drift['status']} (shift: {drift['shift_pct']:.2f}%)")

#  FALSE POSITIVE TESTING
print("\n FALSE POSITIVE TESTING")

scenarios = {'light': (0, 25), 'normal': (25, 75), 'heavy': (75, 99), 'edge': (99, 100)}
results = {}

for name, (p_low, p_high) in scenarios.items():
    min_err = np.percentile(errors_benign_test, p_low)
    max_err = np.percentile(errors_benign_test, p_high)
    mask = (errors_benign_test >= min_err) & (errors_benign_test <= max_err)
    sample = errors_benign_test[mask]

    flagged = (sample >= threshold_ae).sum()
    fp_rate = 100 * flagged / len(sample) if len(sample) > 0 else 0
    results[name] = {'tested': len(sample), 'flagged': flagged, 'fp_rate': fp_rate}
    print(f"  {name}: {flagged}/{len(sample)} FPs ({fp_rate:.2f}%)")

total_fp = sum(r['flagged'] for r in results.values())
total_tested = sum(r['tested'] for r in results.values())
overall_fp = 100 * total_fp / total_tested

print(f"Overall FP rate: {overall_fp:.2f}%")

# SAVE FILES
print("\nSaving files...")
# Saved files:
# 1. ae_normalization_params.json - load this to normalize errors
# 2. drift_detector.py -  import this for monthly monitoring
# 3. fp_analysis_report.txt - Documentation for stakeholder
# File 1: Normalization parameters
params = {
    'p0': float(p0),
    'p99': float(p99),
    'threshold': float(threshold_ae),
    'normalized_threshold': float(normalize_score(threshold_ae)),
    'roc_auc': float(roc_auc),
    'fpr_pct': float(fp_final),
    'tpr_pct': float(tp_final),
}
with open('ae_normalization_params.json', 'w') as f:
    json.dump(params, f, indent=2)
# 1. ae_normalization_params.json - load this to normalize errors

# File 2: Drift detector
drift_code = '''from collections import deque
import numpy as np

class DriftMonitor:
    """Monitor if model behavior is drifting"""
    def __init__(self, baseline_errors, window_size=1000):
        self.baseline_mean = np.mean(baseline_errors)
        self.errors = deque(maxlen=window_size)

    def add_error(self, error):
        self.errors.append(error)

    def check_drift(self):
        if len(self.errors) < 100:
            return {"status": "insufficient_data"}

        current_mean = np.mean(self.errors)
        shift = 100 * (current_mean - self.baseline_mean) / self.baseline_mean

        if abs(shift) < 5:
            status = "normal"
        elif abs(shift) < 15:
            status = "gradual_drift"
        else:
            status = "sudden_drift"

        return {"status": status, "shift_pct": shift}
'''
with open('drift_detector.py', 'w') as f:
    f.write(drift_code)
# 2. drift_detector.py -  import this for monthly monitoring
# File 3: False positive report
report = f"""FALSE POSITIVE ANALYSIS REPORT

Test Scenarios:
  Light traffic (P0-P25):   {results['light']['flagged']}/{results['light']['tested']} FPs ({results['light']['fp_rate']:.2f}%)
  Normal traffic (P25-P75): {results['normal']['flagged']}/{results['normal']['tested']} FPs ({results['normal']['fp_rate']:.2f}%)
  Heavy traffic (P75-P99):  {results['heavy']['flagged']}/{results['heavy']['tested']} FPs ({results['heavy']['fp_rate']:.2f}%)
  Edge case (P99-P100):     {results['edge']['flagged']}/{results['edge']['tested']} FPs ({results['edge']['fp_rate']:.2f}%)

Overall: {total_fp}/{total_tested} false positives ({overall_fp:.2f}%)
Verdict: {'PASS' if overall_fp < 5 else 'FAIL'}

Interpretation: {overall_fp:.2f}% of legitimate traffic would be flagged as attacks.
Acceptable if < 5%. Current threshold ({threshold_ae:.6f}) is {'ready for production' if overall_fp < 5 else 'needs adjustment'}.
"""
with open('fp_analysis_report.txt', 'w') as f:
    f.write(report)
# 3. fp_analysis_report.txt - Documentation for stakeholders

# Summary
print("\nSUMMARY")
print(f"Threshold: {threshold_ae:.6f} | ROC-AUC: {roc_auc:.4f} | FPR: {overall_fp:.2f}%")

#further workflow
# 1. Load ae_normalization_params.json
# 2. Import drift_detector.py for monitoring
# 3. Reference fp_analysis_report.txt for stakeholders

# Saved files:
# 1. ae_normalization_params.json - load this to normalize errors
# 2. drift_detector.py -  import this for monthly monitoring
# 3. fp_analysis_report.txt - Documentation for stakeholders


Data loaded: train=467596 test=116899

 THRESHOLD OPTIMIZATION
ROC-AUC: 0.9635
  P99 Percentile: 0.113518 | FPR: 1.04% TPR: 69.03%
  ROC Optimal: 0.045548 | FPR: 3.97% TPR: 83.88%
  Your Threshold: 0.073801 | FPR: 2.00% TPR: 75.75%
Conclusion: Threshold 0.073801 is optimal

 SCORE NORMALIZATION
Normalization: (error - 0.001261) / (0.113518 - 0.001261)
  Benign mean: 0.0975
  Attack mean: 1.0000
  Threshold score: 0.6462

 DRIFT MONITORING
Status: SUDDEN_DRIFT (shift: -99.39%)

 FALSE POSITIVE TESTING
  light: 0/12401 FPs (0.00%)
  normal: 0/24801 FPs (0.00%)
  heavy: 495/11905 FPs (4.16%)
  edge: 497/497 FPs (100.00%)
Overall FP rate: 2.00%

Saving files...

SUMMARY
Threshold: 0.073801 | ROC-AUC: 0.9635 | FPR: 2.00%
